## SQL Queries to Create Tables

# Orders_1m
```
DROP TABLE IF EXISTS Orders_1m;

CREATE TABLE Orders_1m (
  id SERIAL PRIMARY KEY,
  customer_name TEXT,
  product_category TEXT,
  quantity INTEGER,
  price_per_unit FLOAT,
  order_date DATE,
  country TEXT
);

COPY Orders_1m(customer_name, product_category, quantity, price_per_unit, order_date, country)
FROM '/data/orders_1M.csv'
WITH (FORMAT csv, HEADER true, DELIMITER ',');
```

# People Big
```
DROP TABLE IF EXISTS people_big;

CREATE TABLE people_big (
  id SERIAL PRIMARY KEY,
  first_name TEXT,
  last_name TEXT,
  gender TEXT,
  department TEXT,
  salary INTEGER,
  country TEXT
);

COPY people_big(first_name,last_name,gender,department,salary,country)
FROM '/data/people_1M.csv'
WITH (FORMAT csv, HEADER true, DELIMITER ',');
```

## Query's

# A:
*Which order has the highest price_per_unit?*
```
SELECT oc.customer_name,oc.quantity,oc.price_per_unit
FROM orders_1m oc
ORDER BY price_per_unit DESC
LIMIT 1
```
* Results:

| customer\_name | quantity | price\_per\_unit |
| :--- | :--- | :--- |
| Emma Brown | 3 | 2000 |

# B:
*What are the top 3 product categories with the highest total quantity sold across all orders?*
```
SELECT oc.product_category,sum(oc.quantity) as Y
FROM orders_1m oc
GROUP BY product_category,quantity
ORDER BY Y DESC
LIMIT 3

```
* Results:

| product\_category | y |
| :--- | :--- |
| Office Supplies | 101275 |
| Health & Beauty | 101200 |
| Books | 101065 |

# C:
*What is the total revenue per product category? (Revenue = price_per_unit × quantity)*
```
SELECT oc.product_category ,round(sum(oc.quantity * oc.price_per_unit)::numeric, 2) as TR
FROM orders_1m oc
GROUP BY product_category,quantity
ORDER BY TR ASC
```
* Results:
| product\_category | tr |
| :--- | :--- |
| Books | 854589.11 |
| Grocery | 1016806.93 |
| Toys | 1561414.07 |
| Books | 1715089.82 |
| Grocery | 2042986.08 |
| Fashion | 2089140.19 |
| Books | 2529815.19 |
| Office Supplies | 2573292.67 |
| Grocery | 3078087.06 |
| Health & Beauty | 3098348.03 |
| Toys | 3108082.12 |
| Books | 3377764.32 |
| Grocery | 4061551.24 |
| Sports | 4152920.54 |
| Fashion | 4247023.74 |
| Books | 4254717.6 |
| Toys | 4658894.88 |
| Office Supplies | 5003677.9 |
*Continues a few more lines*

# D:
*Who are the top 5 customers by total spending?*
```
SELECT oc.customer_name ,round(sum(oc.quantity * oc.price_per_unit)::numeric, 2) as TS
FROM orders_1m oc
GROUP BY oc.customer_name
ORDER BY TS DESC
limit 5
```
* Results:
| customer\_name | ts |
| :--- | :--- |
| Carol Taylor | 991179.18 |
| Nina Lopez | 975444.95 |
| Daniel Jackson | 959344.48 |
| Carol Lewis | 947708.57 |
| Daniel Young | 946030.14 |


# E:
*Look at the spending totals in D — and at how many orders each of those customers has. What do you notice? Open ecommerce/dataset_generator.py and explain why the data looks like this.*
```
SELECT oc.customer_name, sum(oc.quantity) AS Norders
FROM orders_1m oc
WHERE customer_name IN
    (SELECT oc.customer_name
    FROM orders_1m oc
    GROUP BY oc.customer_name
    ORDER BY round(sum(oc.quantity * oc.price_per_unit)::numeric, 2) DESC
    limit 5)
GROUP BY oc.customer_name
ORDER BY Norders DESC
```
* Results:
| customer\_name | norders |
| :--- | :--- |
| Daniel Jackson | 3128 |
| Carol Taylor | 3098 |
| Nina Lopez | 2972 |
| Daniel Young | 2915 |
| Carol Lewis | 2899 |

* What i notice it's that they have very similar number of orders , in real life there is usually one or two clients that stand out , this can be explain because some of the data is generatet at random in a range


# Exercise 2

| rows in table    | join result (COUNT(*)) | time   |
|:-----------------|:---:|-------:|
| 50 000           | 166707070  | 9 s 168 ms   |
| 100 000          | 666739290  | 35 s 527 ms |
| 200 000          | 2666831054  | 2 m 8 s 47 ms |
| 100 000 Rewriten | 666739290  | 448 ms |


## Analysis

*based on my calculations doubling the value of n increase by 4 the value of count ( memory ) and time both of them , it takes 4 times more*

O(4n) , O(4n) , taking into acount that is 1M and that is 5 times 200.000 then it would take 20 times as much time and memory to process , the count would be arround

*after using an index we notice that doesn't help , this is because indexes are to help looking for values but in this case the join , joins all the values from the table to each other including themselves , therefore the index doesn't help here.*

*after creating a new query , we can make a table made by the count of each country and then multiply that count by itself, and we get the same values that doing the join but with a fraction of the time , the joint was an unnecessary and inefficient operation in this case.*

## Questions:
* what the rewrite in Step 3 tells you about adding more hardware or an index;
* what you would do if the business actually needed the pairs themselves (not just their count) — would a bigger machine or a cluster help, and how much?
* the limits of an OLTP database for this workload, especially in a large-scale cloud environment.

## Answers

* the rewrite Step tells me that for some problems is possible to optimize the process so much that adding more hardware or an index is totally unnecessary and sometimes useless in the case of an index for a problem that does not search for any value

* in the case that we actually need the all the pairs there is no way that's more efficient , Increasing the size of the machine would allow us to process linearly more data in parallel so is a lineal improvement , however clustering can be even slower because some of the steps are actually faster however there are steps that are even slower in clustering , we have to first cluster that's an extra cost and put all the combined pairs together in a simgle table that's much slower

* because databases OLTP like postgres are build to work with small transactions ,Online Transaction Processing are made to receive many small orders or inserts.However, OLAP databases are build to do actually process big amounts of data

# Rewrite query
```
SELECT Sum(cnt * cnt ) as Pairs
FROM (
SELECT country , count(country) as cnt
From people_100k
GROUP BY country ) t;
```